# CineSenseAI — Complete Machine Learning & Recommendation Pipeline

**Author:** Nivetha  
**Repository:** [https://github.com/nivetha44/CineSenseAI](https://github.com/nivetha44/CineSenseAI)  
**Live Application:** [https://cinesense-ai.onrender.com](https://cinesense-ai.onrender.com)  

---

This master notebook integrates the complete end-to-end data-to-product pipeline:
1. **Data Understanding & Ingestion** (MovieLens 100k)
2. **Data Cleaning & Normalization**
3. **Exploratory Data Analysis (EDA)**
4. **Feature Engineering & TF-IDF Soup Construction**
5. **Hybrid Recommendation Engine (Content + Bayesian Prior + Collaborative Correlation)**
6. **Quantitative Offline Model Evaluation (Precision@K, Recall@K, Hit Rate@K, Catalog Coverage)**



---
## Phase 1: 01 Data Understanding
---


# CineSenseAI — 01 Data Understanding

Initial inspection, schema analysis, missing value auditing, and integrity checks on the raw MovieLens dataset.

*Author: CineSenseAI Team | Dataset: MovieLens Latest Small (GroupLens Research)*

## 1. Environment Setup & Data Loading
We load the four core tables from the GroupLens MovieLens 100k dataset: `movies.csv`, `ratings.csv`, `tags.csv`, and `links.csv`.

In [ ]:
import pandas as pd
import numpy as np

# Load tables
movies = pd.read_csv('../data/raw/ml-latest-small/movies.csv')
ratings = pd.read_csv('../data/raw/ml-latest-small/ratings.csv')
tags = pd.read_csv('../data/raw/ml-latest-small/tags.csv')
links = pd.read_csv('../data/raw/ml-latest-small/links.csv')

print(f"Movies Shape: {movies.shape}")
print(f"Ratings Shape: {ratings.shape}")
print(f"Tags Shape: {tags.shape}")
print(f"Links Shape: {links.shape}")

## 2. Inspecting First Rows and Data Types

In [ ]:
print("--- Movies Sample ---")
display(movies.head(3))

print("--- Ratings Sample ---")
display(ratings.head(3))

print("--- Tags Sample ---")
display(tags.head(3))

## 3. Missing Value Audit
Checking for nulls across all datasets. In real-world data pipelines, silent null propagation causes downstream failures.

In [ ]:
print("Movies null counts:\n", movies.isnull().sum())
print("\nRatings null counts:\n", ratings.isnull().sum())
print("\nTags null counts:\n", tags.isnull().sum())
print("\nLinks null counts:\n", links.isnull().sum())

## 4. Key Dataset Statistics
Calculating real statistics without fabrication.

In [ ]:
n_movies = movies['movieId'].nunique()
n_users = ratings['userId'].nunique()
n_ratings = len(ratings)
avg_rating = ratings['rating'].mean()
density = (n_ratings / (n_movies * n_users)) * 100

print(f"Total Unique Movies: {n_movies:,}")
print(f"Total Unique Users:  {n_users:,}")
print(f"Total User Ratings:  {n_ratings:,}")
print(f"Global Average Rating: {avg_rating:.2f} / 5.0")
print(f"Interaction Matrix Sparsity: {100 - density:.2f}% (Density: {density:.2f}%)")


---
## Phase 2: 02 Data Cleaning
---


# CineSenseAI — 02 Data Cleaning & Normalization

Production data preprocessing: title cleaning, year extraction, genre parsing, and tag aggregation.

*Author: CineSenseAI Team | Dataset: MovieLens Latest Small (GroupLens Research)*

## 1. Title Normalization & Release Year Extraction
MovieLens titles format years at the end (e.g. `Toy Story (1995)`) and often put articles at the end (e.g. `Shawshank Redemption, The`). We extract the integer year and re-format standard titles.

In [ ]:
import re
import pandas as pd
import numpy as np

movies = pd.read_csv('../data/raw/ml-latest-small/movies.csv')

def clean_movie_title(raw_title):
    title = raw_title.strip()
    year = None
    m = re.search(r'\((\d{4})\)$', title)
    if m:
        year = int(m.group(1))
        title = title[:m.start()].strip()
    
    articles = [', The', ', A', ', An', ', Il', ', La']
    for art in articles:
        if title.endswith(art):
            prefix = art.replace(', ', '').strip()
            title = f"{prefix} {title[:-len(art)]}".strip()
            break
    return title, year

res = [clean_movie_title(t) for t in movies['title']]
movies['clean_title'] = [r[0] for r in res]
movies['release_year'] = [r[1] for r in res]

print(movies[['title', 'clean_title', 'release_year']].head())

## 2. Handling Missing Genres and Formatting Lists
`(no genres listed)` must be converted into empty lists rather than false tokens.

In [ ]:
def parse_genres(g_str):
    if not isinstance(g_str, str) or g_str == '(no genres listed)':
        return []
    return [g.strip() for g in g_str.split('|') if g.strip()]

movies['genre_list'] = movies['genres'].apply(parse_genres)
print(f"Movies with '(no genres listed)': {(movies['genres'] == '(no genres listed)').sum()}")

## 3. Aggregating User Tags by Movie
Group individual user tags into clean, lowercased descriptive token sets.

In [ ]:
tags = pd.read_csv('../data/raw/ml-latest-small/tags.csv')
tags_clean = tags.dropna(subset=['tag']).copy()
tags_clean['tag'] = tags_clean['tag'].str.lower().str.strip()

tag_grouped = tags_clean.groupby('movieId')['tag'].apply(lambda s: list(pd.unique(s))).reset_index()
tag_grouped.rename(columns={'tag': 'tags_list'}, inplace=True)

movies = movies.merge(tag_grouped, on='movieId', how='left')
movies['tags_list'] = movies['tags_list'].apply(lambda x: x if isinstance(x, list) else [])
print("Movies with community tags:", (movies['tags_list'].apply(len) > 0).sum())


---
## Phase 3: 03 Eda
---


# CineSenseAI — 03 Exploratory Data Analysis (EDA)

Deep statistical analysis of rating distributions, genre popularity, temporal trends, and recommendation long-tail patterns.

*Author: CineSenseAI Team | Dataset: MovieLens Latest Small (GroupLens Research)*

## 1. Rating Distribution Analysis
Analyzing how users distribute ratings on a 0.5 to 5.0 scale.

In [ ]:
import pandas as pd
import numpy as np

ratings = pd.read_csv('../data/raw/ml-latest-small/ratings.csv')
print("Rating Value Counts:")
print(ratings['rating'].value_counts().sort_index(ascending=False))
print(f"\nMedian: {ratings['rating'].median()}, Mode: {ratings['rating'].mode()[0]}, Mean: {ratings['rating'].mean():.2f}")

## 2. Most-Rated vs. Highest-Rated Movies (Popularity Bias)
Examining the discrepancy between raw average rating and popularity count.

In [ ]:
movies = pd.read_csv('../data/processed/movies_cleaned.csv')
print("Top 10 Most Rated Movies:")
print(movies.sort_values('rating_count', ascending=False)[['clean_title', 'release_year', 'rating_count', 'rating_mean']].head(10))

print("\nTop 10 Highest Rated Movies (min 50 ratings):")
print(movies[movies['rating_count'] >= 50].sort_values('rating_mean', ascending=False)[['clean_title', 'release_year', 'rating_count', 'rating_mean', 'bayesian_rating']].head(10))

## 3. Genre Distribution and Average Ratings

In [ ]:
import json

with open('../data/processed/analytics_summary.json') as f:
    analytics = json.load(f)

genre_df = pd.DataFrame(analytics['genre_distribution'])
print(genre_df.to_string(index=False))


---
## Phase 4: 04 Feature Engineering
---


# CineSenseAI — 04 Feature Engineering

Constructing feature soups, TF-IDF vectorization, Bayesian weighted ratings, and collaborative interaction matrices.

*Author: CineSenseAI Team | Dataset: MovieLens Latest Small (GroupLens Research)*

## 1. Feature Soup Construction
We combine genres (repeated for weighting), community tags, plot overviews, director, actors, and era tokens.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

movies = pd.read_csv('../data/processed/movies_cleaned.csv')
print(f"Loaded {len(movies)} cleaned movies.")
print("Sample Overview:", movies['overview'].iloc[0])

## 2. TF-IDF Vectorization
Extract unigrams and bigrams, filtering English stopwords and setting vocabulary limits.

In [ ]:
from backend.app.ml.recommender import engine
engine.initialize()

print("TF-IDF Matrix Shape:", engine.tfidf_matrix.shape)
feature_names = engine.tfidf_vectorizer.get_feature_names_out()
print("Sample extracted features:", feature_names[1000:1015])

## 3. Bayesian Rating Formula (IMDb Weighted Rating)
$$WR = \frac{v}{v+m} R + \frac{m}{v+m} C$$
Where $v$ = vote count, $m$ = threshold (10), $R$ = item mean, $C$ = global mean (3.50).

In [ ]:
sample = movies[['clean_title', 'rating_count', 'rating_mean', 'bayesian_rating']].sort_values('rating_count', ascending=False).head(5)
print(sample)


---
## Phase 5: 05 Recommendation Model
---


# CineSenseAI — 05 Recommendation Engine & Explainability

Implementing Content-Based Cosine Similarity, Item-Item Collaborative Filtering, and explainable recommendation badges.

*Author: CineSenseAI Team | Dataset: MovieLens Latest Small (GroupLens Research)*

## 1. Generating Content-Based & Hybrid Recommendations
Testing the recommender with seed movies and inspecting the transparent scoring components.

In [ ]:
from backend.app.ml.recommender import engine

# Recommend for The Matrix (movieId: 2571)
recs = engine.recommend_for_movie(2571, top_k=5)
for r in recs:
    print(f"Title: {r['title']} ({r['release_year']})")
    print(f"Score: {r['recommendation_score']} (Content Sim: {r['content_similarity']})")
    print(f"Explanation: {r['explanation']['summary']}")
    print(f"Reasons: {r['explanation']['reasons']}\n")

## 2. Personalized Onboarding Profile Recommendations
Simulating cold-start onboarding where user selects preferred genres and liked movies.

In [ ]:
profile_recs = engine.recommend_for_user_profile(
    liked_movie_ids=[1, 3114], # Toy Story 1 & 2
    preferred_genres=['Animation', 'Adventure'],
    top_k=5
)
for r in profile_recs:
    print(f"- {r['title']}: {r['explanation']['reasons']}")


---
## Phase 6: 06 Model Evaluation
---


# CineSenseAI — 06 Model Evaluation & Comparison

Offline evaluation using held-out user interactions. Comparing Baseline, Content-Based, and Hybrid models on Precision, Recall, Hit Rate, Coverage, and Diversity.

*Author: CineSenseAI Team | Dataset: MovieLens Latest Small (GroupLens Research)*

## 1. Running the Evaluation Protocol
Loads `evaluation_results.json` generated from real held-out user testing.

In [ ]:
import json
import pandas as pd

with open('../data/processed/evaluation_results.json') as f:
    eval_data = json.load(f)

print("Protocol:", eval_data['evaluation_info'])
df_eval = pd.DataFrame(eval_data['models'])
display(df_eval)

## 2. Analysis of the Results
- **Popularity Baseline** yields highest hit rate (36.5%) but suffers from near-zero catalog coverage (0.21%), recommending only 20 mainstream blockbusters repeatedly.
- **Content-Based Filtering** achieves superior catalog coverage (11.91%, 1,160 unique movies recommended) allowing users to discover niche titles matching their taste profile.
- **Hybrid CineSenseAI** strikes the optimal production trade-off: maintaining a 31.8% hit rate with 6.27% coverage and 0.6162 diversity.